# ERA5 climate data - SNT sync report

In [ ]:
# Configuration
ROOT_PATH <- "~/workspace"
CODE_PATH <- file.path(ROOT_PATH, "code")
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
DATA_PATH <- file.path(ROOT_PATH, "data")
ERA5_AGGREGATE_PATH <- file.path(DATA_PATH, "era5", "aggregate")
PLOTS_PATH <- file.path(ROOT_PATH, "pipelines", "snt_era5_sync", "reporting", "outputs")

source(file.path(CODE_PATH, "snt_utils.r"))
install_and_load(c("dplyr", "tidyr", "ggplot2", "glue", "arrow", "sf", "reticulate", "lubridate", "viridis", "jsonlite"))

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
openhexa <- reticulate::import("openhexa.sdk")
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

config_json <- jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

dir.create(PLOTS_PATH, recursive = TRUE, showWarnings = FALSE)
log_msg(glue::glue("ERA5 sync report for {COUNTRY_CODE}. Outputs: {PLOTS_PATH}"))

In [ ]:
# Load monthly aggregates by variable
if (!dir.exists(ERA5_AGGREGATE_PATH)) {
  stop(glue::glue("ERA5 aggregate path not found: {ERA5_AGGREGATE_PATH}"))
}

variable_dirs <- list.dirs(ERA5_AGGREGATE_PATH, full.names = TRUE, recursive = FALSE)
variables <- basename(variable_dirs)

monthly_data <- list()
for (var in variables) {
  f <- file.path(ERA5_AGGREGATE_PATH, var, glue::glue("{COUNTRY_CODE}_{var}_monthly.parquet"))
  if (file.exists(f)) {
    monthly_data[[var]] <- arrow::read_parquet(f) %>% dplyr::mutate(VARIABLE = var)
    log_msg(glue::glue("Loaded {var}: {nrow(monthly_data[[var]])} rows"))
  }
}

if (length(monthly_data) == 0) {
  stop("No monthly ERA5 data found. Run snt_era5_sync first.")
}

In [ ]:
# National monthly temperature plot (if available)
var_temp <- "2m_temperature"
if (var_temp %in% names(monthly_data)) {
  d <- monthly_data[[var_temp]] %>% dplyr::mutate(PERIOD_DATE = lubridate::ym(as.character(PERIOD)))
  national <- d %>% dplyr::group_by(PERIOD, PERIOD_DATE) %>% dplyr::summarise(MEAN = mean(MEAN, na.rm = TRUE), .groups = "drop")

  p_temp <- ggplot2::ggplot(national, ggplot2::aes(x = PERIOD_DATE, y = MEAN)) +
    ggplot2::geom_line(linewidth = 0.6, color = "#c62828") +
    ggplot2::theme_minimal() +
    ggplot2::labs(title = glue::glue("Temperature monthly mean (ERA5) - {COUNTRY_CODE}"), x = "Period", y = "Temperature (C)")
  print(p_temp)

  ggplot2::ggsave(file.path(PLOTS_PATH, glue::glue("{COUNTRY_CODE}_era5_temperature_monthly.png")), p_temp, width = 10, height = 4, dpi = 150)
}

In [ ]:
# National monthly precipitation plot
var_precip <- "total_precipitation"
if (var_precip %in% names(monthly_data)) {
  d <- monthly_data[[var_precip]] %>% dplyr::mutate(PERIOD_DATE = lubridate::ym(as.character(PERIOD)))
  national <- d %>% dplyr::group_by(PERIOD, PERIOD_DATE) %>% dplyr::summarise(MEAN = mean(MEAN, na.rm = TRUE), .groups = "drop")

  p_precip <- ggplot2::ggplot(national, ggplot2::aes(x = PERIOD_DATE, y = MEAN)) +
    ggplot2::geom_line(linewidth = 0.6, color = "#1565c0") +
    ggplot2::theme_minimal() +
    ggplot2::labs(title = glue::glue("Precipitation monthly mean (ERA5) - {COUNTRY_CODE}"), x = "Period", y = "Precipitation (mm)")
  print(p_precip)

  ggplot2::ggsave(file.path(PLOTS_PATH, glue::glue("{COUNTRY_CODE}_era5_precipitation_monthly.png")), p_precip, width = 10, height = 4, dpi = 150)
}

In [ ]:
# Load shapes and create annual precipitation maps
spatial_data <- tryCatch({
  get_latest_dataset_file_in_memory(dhis2_dataset, glue::glue("{COUNTRY_CODE}_shapes.geojson"))
}, error = function(e) {
  log_msg(glue::glue("Could not load shapes for maps: {conditionMessage(e)}"), level = "warning")
  NULL
})

if (!is.null(spatial_data) && var_precip %in% names(monthly_data)) {
  precip_breaks <- c(0, 99, 249, 799, 1199, Inf)
  precip_labels <- c("0-99", "100-249", "250-799", "800-1199", ">=1200")
  precip_colors <- c("#F7FBFF", "#C6DBEF", "#6BAED6", "#2171B5", "#084594")
  names(precip_colors) <- precip_labels

  annual_by_year <- monthly_data[[var_precip]] %>%
    dplyr::group_by(ADM2_ID, YEAR) %>%
    dplyr::summarise(TOTAL_MM = sum(MEAN, na.rm = TRUE), .groups = "drop") %>%
    dplyr::mutate(PRECIP_CAT = cut(TOTAL_MM, breaks = precip_breaks, labels = precip_labels, include.lowest = TRUE, right = FALSE))

  plot_grid <- spatial_data %>% dplyr::left_join(annual_by_year, by = "ADM2_ID")

  p_grid <- ggplot2::ggplot(plot_grid) +
    ggplot2::geom_sf(ggplot2::aes(fill = PRECIP_CAT), color = "black", linewidth = 0.1) +
    ggplot2::scale_fill_manual(values = precip_colors, na.value = "grey95", drop = FALSE) +
    ggplot2::facet_wrap(~ YEAR, ncol = 4) +
    ggplot2::theme_void() +
    ggplot2::theme(legend.position = "bottom") +
    ggplot2::labs(title = "Annual precipitation (ERA5)")
  print(p_grid)

  ggplot2::ggsave(file.path(PLOTS_PATH, glue::glue("{COUNTRY_CODE}_era5_precipitation_annual_grid.png")), p_grid, width = 12, height = 8, dpi = 200)
}

log_msg(glue::glue("ERA5 sync report finished. Plots saved in {PLOTS_PATH}"))